In [1]:
from data.preprocessing.pipeline import Pipeline

In [2]:
pipeline = Pipeline(download=False)

Extracting raw features: 100%|██████████| 888/888 [00:02<00:00, 399.98it/s]


In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.inspection import permutation_importance
import optuna

from catboost import CatBoostClassifier

/home/smirnov@dohod.local/Desktop/thortex-hackaton/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# -------------------------------------------------------
# 1. Загрузка данных
# -------------------------------------------------------
train_df, test_df = pipeline.get_processed_data()

y = train_df["taste_cluster"].values
feature_cols = [c for c in train_df.columns if c.startswith("feat_")]
X_raw = train_df[feature_cols].values
ids = train_df["Pubchem_ID"].values

# Логарифмирование фич
X = np.log1p(X_raw)

# -------------------------------------------------------
# 2. Split с сохранением Pubchem_ID
# -------------------------------------------------------
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X, y, ids,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# -------------------------------------------------------
# 3. Черновой CatBoost → отбор признаков
# -------------------------------------------------------
print("\n>>> Черновой CatBoost для отбора фич...")

tmp_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=False,
    random_seed=42,
    task_type="GPU"    # если есть GPU
)

tmp_model.fit(X_train, y_train)

print("\n>>> Расчёт permutation importance...")

perm = permutation_importance(
    tmp_model, X_val, y_val,
    scoring="f1_macro",
    n_repeats=5,
    random_state=42
)

perm_importances = perm.importances_mean
threshold = np.quantile(perm_importances, 0.25)  # оставляем лучшие 75%

mask = perm_importances > threshold

print(f"Используем {mask.sum()} признаков из {len(mask)}")

X_train_red = X_train[:, mask]
X_val_red = X_val[:, mask]


# -------------------------------------------------------
# 4. Class weights
# -------------------------------------------------------
cnt = Counter(y_train)
total = len(y_train)
class_weights = {cls: total / cnt[cls] for cls in cnt}

print("\nВеса классов:")
print(class_weights)

/tmp/ipykernel_132168/3990770103.py:12: RuntimeWarning: invalid value encountered in log1p
  X = np.log1p(X_raw)



>>> Черновой CatBoost для отбора фич...


In [ ]:
def objective(trial):
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "iterations": trial.suggest_int("iterations", 1500, 4000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.03),
        "depth": trial.suggest_int("depth", 7, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 3.0, 15.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 5.0),
        "grow_policy": "Lossguide",
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "one_hot_max_size": 2,
        "random_seed": 42,
        "class_weights": class_weights,
        "verbose": False,
        "task_type": "GPU"
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train_red, y_train, eval_set=(X_val_red, y_val), verbose=False)

    preds = model.predict(X_val_red).astype(int).flatten()
    f1 = f1_score(y_val, preds, average="macro")
    return f1

In [ ]:
RUN_OPTUNA = True

if RUN_OPTUNA:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)

    print("Best params:", study.best_params)
    best_params = study.best_params
else:
    best_params = {
        "iterations": 3000,
        "learning_rate": 0.015,
        "depth": 10,
        "l2_leaf_reg": 8,
        "random_strength": 2.5,
        "subsample": 0.85
    }

In [ ]:
model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    grow_policy="Lossguide",
    one_hot_max_size=2,
    bootstrap_type="Bernoulli",
    class_weights=class_weights,
    random_seed=42,
    task_type="GPU",
    verbose=False,
    **best_params
)

print("\n>>> Обучение финальной модели...")
model.fit(X_train_red, y_train, eval_set=(X_val_red, y_val), verbose=False)


>>> Обучение финальной модели...


In [ ]:
y_pred = model.predict(X_val_red).astype(int).flatten()

f1_macro = f1_score(y_val, y_pred, average="macro")
print(f"\nValidation F1-macro: {f1_macro:.4f}\n")

print("=== Classification Report ===")
print(classification_report(y_val, y_pred))


Validation F1-macro: 0.4406

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.62      0.54      0.58       106
           1       0.43      0.34      0.38        59
           2       0.40      0.50      0.44        44
           3       0.50      0.35      0.41        81
           4       0.56      0.58      0.57        57
           5       0.29      0.44      0.35         9
           6       0.31      0.65      0.42        17
           7       0.33      0.45      0.38        40

    accuracy                           0.47       413
   macro avg       0.43      0.48      0.44       413
weighted avg       0.49      0.47      0.47       413



In [ ]:
errors_mask = y_pred != y_val
num_errors = errors_mask.sum()

print(f"\nОшибок: {num_errors}")

if num_errors > 0:
    y_proba = model.predict_proba(X_val_red)

    errors_df = pd.DataFrame({
        "Pubchem_ID": ids_val[errors_mask],
        "true": y_val[errors_mask],
        "pred": y_pred[errors_mask],
        "max_proba": y_proba[errors_mask].max(axis=1)
    })

    print(errors_df.head(10))


Ошибок: 220
   Pubchem_ID  true  pred  max_proba
0       10883     4     3   0.452501
1       15619     3     2   0.368518
2     5463228     2     1   0.316068
3    73604801     2     1   0.245447
4       24114     7     3   0.664344
5       11622     0     3   0.214616
6       61030     4     3   0.362997
7       62126     2     0   0.274041
8       11576     7     3   0.434319
9     5283356     1     6   0.534770


In [ ]:
# -------------------------------------------------------
# 8. Применение модели к тестовым данным и сохранение результата
# -------------------------------------------------------
print("\n>>> Применение модели к тестовым данным...")

# 1. Проверяем наличие столбца Pubchem_ID
if "Pubchem_ID" not in test_df.columns:
    raise KeyError("Столбец 'Pubchem_ID' отсутствует в test_df!")

pubchem_ids = test_df["Pubchem_ID"].values

# 2. Готовим тестовые признаки
X_test_raw = test_df[feature_cols].values

# Логарифмирование — так же, как для train
X_test = np.log1p(X_test_raw)

# 3. Отбор признаков — та же маска, что и при тренировке
if "mask" not in globals():
    raise ValueError("Маска признаков (mask) не найдена! Убедись, что Feature Selection уже выполнен.")

X_test_red = X_test[:, mask]

# 4. Предсказания
test_predictions = model.predict(X_test_red).astype(int).flatten()

# 5. Проверка совпадения размеров
assert len(pubchem_ids) == len(test_predictions), \
    f"Несовпадение длины: Pubchem_ID ({len(pubchem_ids)}) vs predictions ({len(test_predictions)})"

# 6. Запись результата
results_df = pd.DataFrame({
    "Pubchem_ID": pubchem_ids,
    "taste_cluster": test_predictions
})

results_df.to_csv("sol.csv", index=False)

print("✅ Результаты сохранены в sol.csv")
print(f"📌 Всего предсказаний: {len(results_df)}")
print("\nПервые 10 строк:")
print(results_df.head(10))



>>> Применение модели к тестовым данным...
✅ Результаты сохранены в sol.csv
📌 Всего предсказаний: 888

Первые 10 строк:
   Pubchem_ID  taste_cluster
0       32594              2
1        7130              0
2        7455              7
3       31226              7
4       17000              0
5       61093              3
6     5354834              6
7       61457              3
8       80455              1
9      638122              4


/tmp/ipykernel_121157/3916042810.py:16: RuntimeWarning: invalid value encountered in log1p
  X_test = np.log1p(X_test_raw)
